# Bloque Residual

Un bloque residual calcula:

`y = F(x) + S(x)`

Usando tensores `N x C x H x W`, `S(x) = x` cuando las formas de entrada y salida coinciden. Si el tamaño espacial o el conteo de canales cambia, `S(x)` debe transformar la entrada. La adición elemento a elemento requiere formas de tensor idénticas.

**Objetivo:** Completar un bloque residual que use automáticamente un atajo de identidad o un atajo de proyección.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(7)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        # TODO 1: Construir la primera convolución.
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        # TODO 2: Construir el atajo (shortcut).
        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                padding=0,
                bias=False,
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        if self.conv1 is None:
            raise NotImplementedError("Complete self.conv1 en __init__.")
        if self.shortcut is None:
            raise NotImplementedError("Complete self.shortcut en __init__.")

        residual = self.conv1(x)
        residual = self.bn1(residual)
        residual = self.relu(residual)
        residual = self.conv2(residual)
        residual = self.bn2(residual)

        # TODO 3: Realizar la suma residual.
        shortcut = self.shortcut(x)
        out = residual + shortcut
        out = self.relu(out)
        return out

In [2]:
x_identity = torch.randn(4, 32, 64, 64)

identity_block = ResidualBlock(
    in_channels=32,
    out_channels=32,
    stride=1,
)

output = identity_block(x_identity)

print(f"Forma de entrada: {x_identity.shape}")

print(f"Forma de salida: {output.shape}")

print(f"Módulo de atajo: {identity_block.shortcut}")

assert output.shape == x_identity.shape
assert isinstance(identity_block.shortcut, nn.Identity)

Forma de entrada: torch.Size([4, 32, 64, 64])
Forma de salida: torch.Size([4, 32, 64, 64])
Módulo de atajo: Identity()


Pregunta de la celda anterior: **¿Por que la entrada se puede sumar directamente a la ruta residual aqui?**
- Porque las dimensiones del tensor de entrada y del tensor residual son idénticas.

In [3]:
x_projection = torch.randn(
    4,
    32,
    64,
    64,
    requires_grad=True,
)

projection_block = ResidualBlock(
    in_channels=32,
    out_channels=64,
    stride=2,
)

output = projection_block(x_projection)

print(f"Forma de entrada: {x_projection.shape}")

print(f"Forma de salida del bloque residual: {output.shape}")

print(f"Módulo de atajo: {projection_block.shortcut}")

expected_shape = torch.Size([4, 64, 32, 32])
assert output.shape == expected_shape

loss = output.mean()
loss.backward()

print(f"Norma del gradiente de entrada: {x_projection.grad.norm().item():.6f}")

Forma de entrada: torch.Size([4, 32, 64, 64])
Forma de salida del bloque residual: torch.Size([4, 64, 32, 32])
Módulo de atajo: Conv2d(32, 64, kernel_size=(1, 1), stride=(2, 2), bias=False)
Norma del gradiente de entrada: 0.001197


Preguntas de la celda anterior:

1. ¿Qué dimensiones cambiaron?
- Cambió el número de canales de 32 a 64 y la resolución espacial de 64x64 a 32x32 por el `stride = 2`.
2. ¿Por qué `nn.Identity()` no es suficiente aquí?
- Porque no se puede sumar un tensor de 32 canales a uno de 64, ya que el atajo necesita una capa de proyección para transformar la entrada y que sus dimensiones coincidan con la salida de la ruta residual.
3. ¿Qué demuestra el hecho de que `x_projection.grad` no sea `None`?
- Demuestra que el bloque residual es diferenciable y que el gradiente puede fluir hacia atrás a través de ambas rutas (la residual y el atajo).